<a href="https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysl21292/Flyrank-MLE-intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


I like that the paper keeps its negative results and flags its own limits. My questions are about how far the evidence can go.

Finding A (Finding #1): growing pages are 37.6% longer and 20% younger than declining pages (3.2K vs 2.3K words, 184 vs 230 days).
- Where does the label come from? "Growing" and "declining" come from 30 days vs the previous 30 days of impressions with a plus or minus 10% cut. That is a short window, so some pages may be labelled by noise. Would the gap stay with a longer window or a stricter cut?
- Does the design carry the claim? It compares group averages, and the paper reports no intervals. If the growing pages come from a few brands that write longer pages, the gap could be a brand effect. A comparison inside each brand, or a split grouped by brand, would test that.
- A pointer from my own data: in my 30,000-page starter file, younger pages (90-179 days) decline more often (63%) than pages 365+ days old (43%), which points the other way. The data and the cut (20%, not 10%) differ, so this is not a contradiction. I would ask how much the age gap depends on the cut and the client mix.

Finding B (Finding #4): 365+ day pages refreshed within 30 days show 3.2x health and 57x impressions.
- Where does the measure come from? Health score includes impressions (30 points) and position (30 points), so more impressions raises health by construction. The two numbers are not independent evidence.
- Does the design carry the claim? It compares refreshed old pages with unrefreshed old pages, not the same page before and after. Pages chosen for a refresh may have been the stronger ones already, and the paper itself notes survivor bias in small cells. Would a matched comparison (similar prior impressions) or a before/after on the same pages keep a large effect?
- What it supports: "refreshed old pages look healthier". Saying refresh is a lever would need an experiment.

In [1]:
import os, sys, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# In Colab: download the repo once. On your computer: go to the repo folder.
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("Flyrank-MLE-intern"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/ysl21292/Flyrank-MLE-intern.git"], check=True)
    os.chdir("Flyrank-MLE-intern")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["declined"] = (df.trend_direction == "down").astype(int)
y = df.declined.values

# Same inputs as my Week-5 model (observed signals only).
num_cols = ["search_volume", "competition", "cpc", "word_count", "char_count", "days_with_impressions",
            "days_with_sessions", "content_age_days", "days_since_last_update", "ctr", "avg_position",
            "engagement_rate", "scroll_rate", "ai_traffic_pct"]
X = df[num_cols].copy()
X["avg_position"] = X.avg_position.replace(0, np.nan)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    X["log_" + c] = np.log1p(df[c])
X = pd.concat([X, pd.get_dummies(df[["content_type", "main_intent", "competition_level"]].fillna("unknown"), dtype=float)], axis=1)

def make_model(kind):
    fill = SimpleImputer(strategy="median", add_indicator=True)
    if kind == "logistic":
        return make_pipeline(fill, StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    return make_pipeline(fill, RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42, n_jobs=-1))

tiebreak = np.random.default_rng(42).random(len(df))
def precision_at_50(score, yt, tb):
    return yt[np.lexsort((tb, -score))[:50]].mean()

def cv_scores(Xc, splits, kind):
    """Out-of-fold AUC and Precision@50, averaged over folds."""
    aucs, p50s, oof = [], [], np.zeros(len(df))
    for tr, te in splits:
        s = make_model(kind).fit(Xc.iloc[tr], y[tr]).predict_proba(Xc.iloc[te])[:, 1]
        oof[te] = s
        aucs.append(roc_auc_score(y[te], s))
        p50s.append(precision_at_50(s, y[te], tiebreak[te]))
    return {"auc": np.mean(aucs), "p50": np.mean(p50s), "p50_min": np.min(p50s), "p50_max": np.max(p50s), "oof": oof}

print(f"{len(df):,} pages | base rate (random picking): {y.mean():.2f} | seeds fixed at 42")

30,000 pages | base rate (random picking): 0.54 | seeds fixed at 42


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


BEFORE: a random 5-fold split, where a client's pages can sit in both train and test. AFTER: a 5-fold split grouped by client, where whole clients are held out. The base rate is 0.54.

| Model | Split | AUC | P@50 |
|---|---|---|---|
| Logistic regression | BEFORE: random | 0.72 | 0.88 |
| Logistic regression | AFTER: grouped by client | 0.66 | 0.80 |
| Random forest | BEFORE: random | 0.78 | 0.96 |
| Random forest | AFTER: grouped by client | 0.67 | 0.74 |

What I see (measured): the random split made the random forest look 22 points better at P@50 (0.96 vs 0.74). That gap is the model partly recognising clients it had already seen. The grouped numbers are the honest ones, and they are the ones I use in ML-08.

In [2]:
from sklearn.model_selection import GroupKFold, StratifiedKFold

random_splits = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y))      # BEFORE: clients mixed
group_splits = list(GroupKFold(5).split(X, y, groups=df.client_id))                       # AFTER: whole clients held out

rows = []
for kind, name in [("logistic", "Logistic regression"), ("forest", "Random forest")]:
    for split_name, splits in [("BEFORE: random split", random_splits), ("AFTER: grouped by client", group_splits)]:
        r = cv_scores(X, splits, kind)
        rows.append({"model": name, "split": split_name, "AUC": round(r["auc"], 2), "P@50": round(r["p50"], 2),
                     "P@50 worst-best fold": f"{r['p50_min']:.2f}-{r['p50_max']:.2f}"})
print(f"Base rate (random picking): {y.mean():.2f}")
print(pd.DataFrame(rows).to_string(index=False))

Base rate (random picking): 0.54
              model                    split  AUC  P@50 P@50 worst-best fold
Logistic regression     BEFORE: random split 0.72  0.88            0.84-0.92
Logistic regression AFTER: grouped by client 0.66  0.80            0.62-0.88
      Random forest     BEFORE: random split 0.78  0.96            0.90-0.98
      Random forest AFTER: grouped by client 0.67  0.74            0.56-0.90


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Checks on my final inputs:
1. Label-derived columns: trend_direction, trend_pct and the last-30 and previous-30-day columns are banned, and the code confirms none are used. Deliberate leak test: I added one column built from those windows and the random forest jumped to AUC 1.00 and P@50 1.00 even on the grouped split. So the test works. I removed the column and kept the honest number.
2. Overlapping windows: the 90-day totals (impressions, clicks, sessions, ctr, active days) contain the same 60 days that define my label. Test without them: AUC falls from 0.66 to 0.57 for logistic regression and from 0.67 to 0.59 for the random forest, and logistic P@50 drops to 0.57, about the base rate of 0.54. So most of my model's skill comes from totals that overlap the label window.
3. Product flags: none are in the data, and none are used.
4. IDs: never inputs. The split is grouped by client.
5. Base rate: printed next to every metric.

Verdict: no leakage from banned columns, but the window overlap is real. My results are directional evidence, not a clean early-warning test. A future-window label (features from the past, outcome from the next 30 days) would fix this.

In [3]:
# 1) Ablation: does the result survive without the 90-day traffic totals? (they overlap the label's 60 days)
overlap_cols = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
                "ctr", "days_with_impressions", "days_with_sessions"]
rows = []
for kind, name in [("logistic", "Logistic regression"), ("forest", "Random forest")]:
    full = cv_scores(X, group_splits, kind)
    without = cv_scores(X.drop(columns=overlap_cols), group_splits, kind)
    rows.append({"model": name, "test": "all inputs (honest split)", "AUC": round(full["auc"], 2), "P@50": round(full["p50"], 2)})
    rows.append({"model": name, "test": "WITHOUT 90-day traffic totals", "AUC": round(without["auc"], 2), "P@50": round(without["p50"], 2)})

# 2) Deliberate leak: add one column built from the label's own windows, then remove it.
X_leak = X.copy()
X_leak["imp_change"] = (df.impressions_last_30d - df.impressions_prev_30d) / (df.impressions_prev_30d + 1)
leak = cv_scores(X_leak, group_splits, "forest")
rows.append({"model": "Random forest", "test": "WITH leaky column (then removed)", "AUC": round(leak["auc"], 2), "P@50": round(leak["p50"], 2)})

print(f"Base rate (random picking): {y.mean():.2f}")
print(pd.DataFrame(rows).to_string(index=False))

BANNED = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
          "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d", "content_id", "client_id"]
print("\nBanned column in the final inputs?", any(c in X.columns for c in BANNED))

Base rate (random picking): 0.54
              model                             test  AUC  P@50
Logistic regression        all inputs (honest split) 0.66  0.80
Logistic regression    WITHOUT 90-day traffic totals 0.57  0.57
      Random forest        all inputs (honest split) 0.67  0.74
      Random forest    WITHOUT 90-day traffic totals 0.59  0.68
      Random forest WITH leaky column (then removed) 1.00  1.00

Banned column in the final inputs? False


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


My boldest sentence (too strong): "The model finds the pages that are declining, and refreshing them will recover their traffic."

Rewritten in safe language: "On this starter file, tested on clients the model had not seen, a simple model ranked pages with an AUC of about 0.66 and Precision@50 of about 0.80 (random picking gets 0.54). Most of that signal comes from 90-day traffic totals that overlap the label window, so this is directional. The ranked list is decision-support for a person who checks the pages. It does not show that updating a page recovers its traffic."

Failure examples (the model's top picks that did not decline): three keyword articles with 3,881 to 19,360 views, a position of about 6 to 9 and almost no clicks (CTR 0.00 to 0.01), scored 0.92 to 0.93. A page with many views and almost no clicks looks like it is leaking views, but these held steady. I think a 30-day change is noisy for such pages, but I have not tested that.

In [4]:
# Three real failures of the honest (grouped) logistic model: top of each fold's list, but the page did NOT decline.
oof = cv_scores(X, group_splits, "logistic")["oof"]
wrong = []
for _, te in group_splits:
    top = te[np.lexsort((tiebreak[te], -oof[te]))[:50]]
    wrong += [i for i in top if y[i] == 0]
wrong = sorted(wrong, key=lambda i: -oof[i])[:3]
print(f"{len(wrong)} failure examples (model score high, page did not decline):")
print(df.loc[wrong, ["content_type", "impressions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update"]].assign(model_score=np.round(oof[wrong], 2)).to_string(index=False))

3 failure examples (model score high, page did not decline):
   content_type  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update  model_score
keyword article            15101           5.7 0.00               421                       7         0.93
keyword article             3881           8.5 0.00               144                     104         0.92
keyword article            19360           6.9 0.01               236                     104         0.92


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.